# Inferência: aplicar os modelos treinados a dados nunca vistos

Este notebook **usa os modelos já treinados** (em `outputs/models/`) para pontuar dados
de **validação** — relatórios OpenFDA que os modelos nunca viram. Ele **não treina nada**:
reaplica exatamente a mesma transformação do `00_final.ipynb` (via o módulo
`src/dataprep.py`) e o **imputador KNN ajustado só no treino**, garantindo **zero
vazamento**.

**Pré-requisitos**
- Ter rodado o `00_final.ipynb` ao menos uma vez (gera `outputs/models/*.joblib`).
- Ter os JSON a pontuar em uma pasta (padrão: `validation/`).
- Para o imputador da faixa etária, ter os JSON de treino em `datasets/` **ou** o
  imputador já salvo em `outputs/models/age_group_knn_imputer.joblib` (este notebook o
  salva na primeira execução).

> **Custo:** pontuar dados brutos exige reingerir e transformar os JSON-alvo (a validação
> tem ~1,2 GB). Use `MAX_RECORDS` para um teste rápido, ou rode numa máquina mais potente
> (ver a seção de execução remota no README).

## 0. Configuração e imports

In [ ]:
import sys
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    ConfusionMatrixDisplay, accuracy_score, balanced_accuracy_score,
    confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score,
)

# Resolve a raiz do projeto (o notebook roda de notebooks/ ou da raiz) e expõe `src`.
_cwd = Path.cwd()
PROJ = _cwd.parent if _cwd.name == "notebooks" else _cwd
sys.path.insert(0, str(PROJ))
from src import dataprep as dp  # noqa: E402

OUTPUTS_DIR = PROJ / "outputs"
MODELS_DIR = OUTPUTS_DIR / "models"
DATASETS_DIR = PROJ / "datasets"

# ---- Configuração da inferência ----
# Modelo a usar (padrão: melhor experimento, EXP-05 Random Forest · Completo).
MODEL_FILE = MODELS_DIR / "EXP-05_random_forest.joblib"
# Pasta com os JSON a pontuar (dados nunca vistos).
DATA_DIR = PROJ / "validation"
# Limite de registros para teste rápido (None = todos).
MAX_RECORDS = None
# Caminho do imputador KNN persistido (reutilizado entre execuções).
IMPUTER_PATH = MODELS_DIR / "age_group_knn_imputer.joblib"

print("Modelo :", MODEL_FILE.name, "| existe:", MODEL_FILE.exists())
print("Dados  :", DATA_DIR, "|", len(list(DATA_DIR.glob("*.json"))), "JSON")
print("MAX_RECORDS:", MAX_RECORDS)

## 1. Imputador KNN da faixa etária (ajustado só no treino)

Carrega o imputador salvo, se existir; caso contrário, ajusta-o nos dados de **treino**
(mesmo split estratificado 70/30 do `00_final`, `random_state=42`) e o persiste. Ajustar
só no treino é o que evita vazamento para os dados de validação.

In [ ]:
if IMPUTER_PATH.exists():
    age_imputer = joblib.load(IMPUTER_PATH)
    print("Imputador KNN carregado de", IMPUTER_PATH.name)
else:
    from sklearn.model_selection import train_test_split
    print("Imputador não encontrado — ajustando no treino (datasets/)...", flush=True)
    raw_train = OUTPUTS_DIR / "_raw_reports_train.parquet"
    prepared = dp.prepare_dataframe(sorted(DATASETS_DIR.glob("*.json")), raw_train)
    prepared = prepared[prepared[dp.TARGET].notna()]
    train_df, _ = train_test_split(
        prepared, test_size=0.30, random_state=42, stratify=prepared[dp.TARGET],
    )
    age_imputer = dp.fit_age_group_imputer(train_df)
    joblib.dump(age_imputer, IMPUTER_PATH)
    print("Imputador KNN ajustado e salvo em", IMPUTER_PATH.name)

## 2. Transformar os dados a pontuar

Ingestão preguiçosa + transformação (mesma do `00_final`), depois imputação da faixa
etária com o imputador do treino.

In [ ]:
raw_infer = OUTPUTS_DIR / "_raw_reports_inference.parquet"
df = dp.prepare_dataframe(sorted(DATA_DIR.glob("*.json")), raw_infer,
                          max_records=MAX_RECORDS)
df = dp.impute_age_groups(df, age_imputer)
print("Registros transformados:", df.shape)
df.head(3)

## 3. Carregar o modelo e prever

As colunas de entrada são lidas do próprio pipeline treinado
(`feature_names_in_` do pré-processador), então o conjunto certo (Baseline ou Completo) é
selecionado automaticamente.

In [ ]:
model = joblib.load(MODEL_FILE)
cols = list(model.named_steps["preprocess"].feature_names_in_)
X = df[cols]

pred = model.predict(X)                 # rótulo (1 = grave, 2 = não-grave)
proba = model.predict_proba(X)[:, 1]    # probabilidade contínua (classe 2)

resultado = df[["safetyreportid"]].copy()
resultado["serious_pred"] = pred
resultado["proba_classe2"] = proba
dest = OUTPUTS_DIR / "predicoes_inferencia.csv"
resultado.to_csv(dest, index=False)
print("Previsões salvas em:", dest, "| linhas:", len(resultado))
resultado.head()

## 4. Métricas (quando há gabarito)

Se os registros trazem o rótulo `serious`, calculamos as mesmas métricas do `00_final`
(rótulo para acurácia/precisão/recall/F1; probabilidade para ROC AUC) — uma medida de
generalização em dados nunca vistos.

In [ ]:
tem_gabarito = df[dp.TARGET].notna().any()
if not tem_gabarito:
    print("Sem coluna 'serious' nos dados — pontuação gerada sem métricas.")
else:
    mask = df[dp.TARGET].notna()
    y = df.loc[mask, dp.TARGET].astype(int)
    yp, ypr = pred[mask.values], proba[mask.values]
    metricas = {
        "n": int(mask.sum()),
        "accuracy": accuracy_score(y, yp),
        "balanced_accuracy": balanced_accuracy_score(y, yp),
        "precision_pos": precision_score(y, yp, pos_label=1, zero_division=0),
        "recall_pos": recall_score(y, yp, pos_label=1, zero_division=0),
        "f1_pos": f1_score(y, yp, pos_label=1, zero_division=0),
        "f1_macro": f1_score(y, yp, average="macro", zero_division=0),
        "roc_auc": roc_auc_score(y, ypr),
    }
    print(MODEL_FILE.name)
    for k, v in metricas.items():
        print(f"  {k:>18}: {v:.4f}" if isinstance(v, float) else f"  {k:>18}: {v}")

    fig, ax = plt.subplots(figsize=(4.5, 4))
    ConfusionMatrixDisplay(confusion_matrix(y, yp), display_labels=[1, 2]).plot(
        ax=ax, cmap="Blues", colorbar=False, values_format="d")
    ax.set_title(f"Matriz de confusão — {MODEL_FILE.stem}")
    ax.set_xlabel("Previsto"); ax.set_ylabel("Real")
    plt.tight_layout(); plt.show()